# METAVCI_COGNITION — Bayesian network analysis

Given the three parquets from `00_preprocess.ipynb`, this notebook:

1. Discretises continuous variables.
2. Learns a layered Bayesian network (arcs constrained by `layer_map`).
3. Renders the structure and exports a PDF (editable in Illustrator).
4. Ranks variables by mutual information with each outcome.
5. Quantifies edge stability via bootstrap resampling.
6. Runs a single-knob sensitivity sweep for a chosen variable.
7. Persists the network as `bn.bifxml`.

Almost everything is a parameter. Tune `outcomes`, `layer_map`,
`base_profile` and the run-time settings without editing `core/`.


## Setup

In [ ]:
# Cell 1 — sys.path bootstrap so `import core.*` works.
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = _here
while _root != _root.parent and not (_root / "core").is_dir():
    _root = _root.parent
if not (_root / "core").is_dir():
    raise RuntimeError(f"Could not locate `core/` above {_here}")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Repo root: {_root}")


In [ ]:
# Cell 2 — imports.
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyagrum as gum
import pyagrum.lib.notebook as gnb

from core.config import load_project_config
from core.io import read_parquet
from core.discretisation import make_type_processor, describe_template
from core.bn_utils import (
    build_bn, bootstrap_edge_frequencies,
    bootstrap_scenario_risks, bootstrap_knob_sweep,
)
from core.inference import mutual_information_scores, conditional_mutual_information_scores
from core.plotting import (
    default_layer_colors, build_node_colors,
    show_and_save_bn, plot_knob_sweep,
)

gum.config["notebook", "graph_layout"] = "dot"
gum.config["notebook", "graph_rankdir"] = "TB"

mpl.rcParams["font.family"] = "Helvetica Neue"
mpl.rcParams["font.size"] = 14


## Load config and data

In [ ]:
# Cell 3 — read config + parquet outputs from 00_preprocess.ipynb.
config = load_project_config()
data_dir = config.output_dir

df      = read_parquet(data_dir / "df.parquet")
df_imp  = read_parquet(data_dir / "df_imp.parquet")
bn_vars = read_parquet(data_dir / "bn_vars.parquet")

# Output folder for BN figures (PDF, editable in Illustrator).
OUTPUT_GRAPHS_DIR = Path(config.project_root) / "outputs" / "graphs"
OUTPUT_GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"df:     {len(df)} rows × {len(df.columns)} cols")
print(f"df_imp: {len(df_imp)} rows × {len(df_imp.columns)} cols")
print(f"layers: {sorted(bn_vars['LAYER'].unique())}")


## Layer map

`layer_map` groups variables by expert-defined layer. The learner
only considers arcs that go **from an earlier layer to the same or a
later layer**. The order below is the order used for that constraint,
so put your input layers first and your outcomes last.

In [ ]:
# Cell 4 — build layer_map (ordered dict; keys are layer names).
layer_map = (
    bn_vars.groupby("LAYER", sort=False)["VARIABLE NAME"]
    .apply(list)
    .to_dict()
)

# TODO: define which columns are the *outcomes* the network should learn.
outcomes = [
    # "OUTCOME_X",
    # "DROPOUT REASON",
]

# TODO: layers to exclude from structure learning (e.g. biomarker layer
# treated separately). Leave empty if all layers participate.
exclude_layers: list[str] = []

for layer, variables in layer_map.items():
    print(f"  {layer:60}  {len(variables)} vars")


## Discretisation

Numeric variables are binned by `DiscreteTypeProcessor`; categorical
variables pass through. `describe_template` returns a DataFrame with
the resulting bins, useful for documenting the discretisation.

In [ ]:
# Cell 5 — configure discretisation.
type_processor = make_type_processor(method="quantile", n_bins=4, threshold=10)
template = type_processor.discretizedTemplate(df_imp)
display(describe_template(template))


## Structure learning

`build_bn` is the workhorse. Its most important knobs:

| Argument | Purpose |
| --- | --- |
| `score` | `"K2"` (default), `"BIC"` (adds smoothing prior), `"BDeu"` |
| `enforce_structure` | If True, apply the layered arc constraints |
| `exclude_layers` | Layer names to drop from learning entirely |
| `max_indegree` | Cap on the number of parents per node |
| `use_smoothing` | Laplace prior — turn on if inference errors mention zero-probability entries |


In [ ]:
# Cell 6 — learn the network.
bn = build_bn(
    df_imp,
    outcomes=outcomes,
    layer_map=layer_map,
    type_processor=type_processor,
    score="K2",
    exclude_layers=exclude_layers,
    random_seed=config.seed,
)

print(f"Learned BN: {bn.size()} nodes, {len(list(bn.arcs()))} arcs")


## Visualisation

In [ ]:
# Cell 7 — colour nodes by layer, then show & save.
layer_order = list(layer_map.keys())
layer_colors = default_layer_colors(layer_order)
node_colors  = build_node_colors(bn, layer_map, layer_colors)

show_and_save_bn(
    bn,
    save_path=OUTPUT_GRAPHS_DIR / "bn_structure.pdf",
    inference=False,
    nodeColor=node_colors,
    cmapNode=plt.get_cmap("coolwarm"),
)


## Bootstrap edge stability

Fits the network on `n_bootstraps` resamples and returns a per-arc
frequency (0–1). Arcs that appear consistently across resamples are
more trustworthy.

In [ ]:
# Cell 8 — bootstrap edge frequencies (adjust n_bootstraps for speed).
edge_freqs = bootstrap_edge_frequencies(
    df_imp,
    outcomes=outcomes,
    layer_map=layer_map,
    type_processor=type_processor,
    n_bootstraps=200,
    random_seed=config.seed,
    exclude_layers=exclude_layers,
    score="K2",
)

edge_freq_df = (
    pd.DataFrame([(p, c, f) for (p, c), f in edge_freqs.items()],
                 columns=["parent", "child", "frequency"])
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)
display(edge_freq_df.head(30))


## Mutual information ranking

For each outcome, which variables carry the most information about
it? `conditional_mutual_information_scores` conditions on the
outcome's parents so you see what a variable adds *beyond* its Markov
blanket.

In [ ]:
# Cell 9 — MI rankings per outcome.
for outcome in outcomes:
    if outcome not in bn.names():
        continue
    print(f"\n=== Mutual information with {outcome} ===")
    mi = mutual_information_scores(bn, outcome)
    display(mi.head(15).to_frame())

    print(f"=== Conditional MI (given parents) — {outcome} ===")
    cmi = conditional_mutual_information_scores(bn, outcome)
    display(cmi.head(15).to_frame())


## Scenario risks

For a list of pre-defined patient profiles, compute the posterior
probability of each outcome state — with a bootstrap 95% CI.

In [ ]:
# Cell 10 — bootstrap scenario risks.
scenario_profiles = [
    # ("Healthy profile", {"AGE": "58", "SEX": "Male", "SYS_BP": "120", ...}),
    # ("Ill profile",     {"AGE": "76", "SEX": "Female", "SYS_BP": "160", ...}),
]

target_outcomes = [
    # ("OUTCOME_X", "Outcome X (display label)"),
]

if scenario_profiles and target_outcomes:
    scenario_summary = bootstrap_scenario_risks(
        df_imp,
        scenario_profiles=scenario_profiles,
        target_outcomes=target_outcomes,
        layer_map=layer_map,
        type_processor=type_processor,
        outcomes_for_learning=outcomes,
        n_bootstraps=200,
        random_seed=config.seed,
        exclude_layers=exclude_layers,
        score="K2",
    )
    display(scenario_summary)
else:
    print("Define scenario_profiles and target_outcomes to run this cell.")


## Single-knob sensitivity sweep

Fix a patient (`base_profile`), sweep one variable (`knob`) across
its discretised states, and plot how each outcome responds — with
bootstrap ribbons. The template uses a fixed discretisation across
resamples so knob states mean the same thing every time.

**Watch out for overadjustment**: if `base_profile` fixes a variable
that lies downstream of `knob`, it blocks part of the knob's
influence on the outcomes. The function warns you when this happens.


In [ ]:
# Cell 11 — single-knob sensitivity.
base_profile = {
    # "AGE": 65, "SEX": "Male",  # numeric values are snapped to the correct bin
}
knob = ""  # e.g. "SMALL VESSEL DISEASE SCORE"

if base_profile and knob:
    sweep_df, meta = bootstrap_knob_sweep(
        df_imp,
        base_profile=base_profile,
        knob=knob,
        outcomes=[o for o in outcomes if "DROPOUT" not in o.upper()],
        outcomes_for_learning=outcomes,
        layer_map=layer_map,
        type_processor=type_processor,
        n_bootstraps=200,
        random_seed=config.seed,
        exclude_layers=exclude_layers,
        score="K2",
    )
    display(sweep_df)
    plot_knob_sweep(sweep_df, meta)
else:
    print("Define base_profile and knob to run this cell.")


## Persist the network

In [ ]:
# Cell 12 — save BIF-XML for external tools.
bn_path = Path(config.project_root) / "bn.bifxml"
gum.saveBN(bn, str(bn_path))
print(f"Saved network to {bn_path}")
